# VynFi GNN fraud-detection showcase — reproducible end-to-end

This notebook reproduces the [`VynFi/je-fraud-gnn`](https://huggingface.co/VynFi/je-fraud-gnn) model bundle from the published [`VynFi/vynfi-journal-entries-1m`](https://huggingface.co/datasets/VynFi/vynfi-journal-entries-1m) dataset (DataSynth v5.9.0, Method-A edges).

The two tasks:
1. **Edge fraud classification** (supervised) — GraphSAGE encoder + edge head MLP.
2. **Edge / node anomaly scoring** (unsupervised) — attribute-reconstruction GAE.

## Reproduce the published bundle

From a clean clone of [`mivertowski/SyntheticData`](https://github.com/mivertowski/SyntheticData):

```bash
pip install -r requirements-ml.txt
python -m scripts.ml.build_je_pyg_dataset --output data/ml/je_pyg_v1.pt --seed 20260509
python -m scripts.ml.train_je_fraud_gnn --epochs 60
python -m scripts.ml.train_je_anomaly_gae --epochs 80
python -m scripts.ml.package_for_hf
```

Determinism: ChaCha8-derived seed `20260509`, sklearn `random_state=0`, torch `manual_seed`. Bit-for-bit reproducible on CPU.

## Inspect the published model bundle

Pull the model from the Hub and run a few sample predictions.

In [ ]:
from huggingface_hub import snapshot_download
from scripts.ml.inference import load_bundle

local_dir = snapshot_download(repo_id='VynFi/je-fraud-gnn')
bundle = load_bundle(local_dir)
print(f'nodes: {bundle.metadata["n_nodes"]}, edges: {bundle.metadata["n_edges"]:,}')
print(f'fraud test AUC : {bundle.metadata["fraud_metrics"]["gnn"]["test"]["auc_roc"]:.4f}')
print(f'fraud test F1  : {bundle.metadata["fraud_metrics"]["gnn"]["test"]["f1"]:.4f}')
print(f'GAE edge AUC   : {bundle.metadata["anomaly_metrics"]["edge_results"]["auc_roc"]:.4f}')

## Single-edge fraud prediction

Try a clear-fraud signature (round amount + weekend) vs a clean non-fraud.

In [ ]:
samples = [
    # Clear fraud — $25K + Saturday
    dict(from_account='1000', to_account='2000', amount=25_000.00,
         business_process='P2P', posting_date='2024-08-10'),
    # Clean — $7,432.89 + Tuesday
    dict(from_account='1000', to_account='2000', amount=7_432.89,
         business_process='P2P', posting_date='2024-03-12'),
    # Borderline — $10K + Wednesday
    dict(from_account='1000', to_account='2000', amount=10_000.00,
         business_process='P2P', posting_date='2024-05-15'),
]

probs = bundle.predict_fraud(
    from_account=[s['from_account'] for s in samples],
    to_account=[s['to_account'] for s in samples],
    amount=[s['amount'] for s in samples],
    business_process=[s['business_process'] for s in samples],
    posting_date=[s['posting_date'] for s in samples],
)
for s, p in zip(samples, probs):
    print(f'{s["from_account"]} -> {s["to_account"]} ${s["amount"]:>10,.2f} {s["posting_date"]} -> fraud_p={p:.4f}')

## Why the GraphSAGE lift over LR is small

DataSynth's `fraud_bias` mechanism injects very strong *local* signals into edge attributes:

| Bias | Probability | Effect |
|---|---|---|
| `weekend_bias` | 30 % | Posting date shifted to Sat/Sun |
| `round_dollar_bias` | 40 % | Amount rescaled to `$1K/$5K/$10K/$25K/$50K/$100K` |
| `off_hours_bias` | 35 % | `created_at` shifted to 22:00–05:59 |
| `post_close_bias` | 25 % | `is_post_close = true` |

The first two land directly on edge attributes that we encode in the feature vector. As a result a vanilla LogisticRegression already gets to AUC 0.91 on the supervised edge task — leaving only +0.13 AUC pts of lift for the GraphSAGE encoder.

Where the graph signal *does* show up:

- **Unsupervised anomaly scoring** — the attribute-reconstruction GAE reaches edge AUC 0.65 *with no labels at train time*.
- **Per-process specificity** — A2R (rare, high-bias) reaches 0.95 supervised AUC vs P2P (more diffuse) at 0.93.
- **Cold-start / multi-hop fraud** — not directly testable on this dataset because the labelling is single-edge-bound, but graph methods are the natural fit.

Where the LR baseline catches up:

- Pure single-transaction fraud detection where the signal is fully in attributes.

## Try the live demos

- 🔗 **Accounting Network Explorer** — interactive ISO 21378 account-class graph: <https://huggingface.co/spaces/VynFi/accounting-network-explorer>
- 🛡️ **Fraud-GNN Demo** — Gradio inference Space (the model from this notebook): <https://huggingface.co/spaces/VynFi/fraud-gnn-demo>

## Citation

```bibtex
@misc{ivertowski2026datasynth,
  author       = {Ivertowski, Michael},
  title        = {{DataSynth}: Reference Knowledge Graphs for Enterprise
                  Audit Analytics through Synthetic Data Generation
                  with Provable Statistical Properties},
  year         = {2026},
  month        = {April},
  howpublished = {SSRN Working Paper},
  url          = {https://ssrn.com/abstract=6538639}
}
```